# Football Match Predictor
Select home and away teams from the dropdowns to get predictions.

In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from scripts import config, data_loader, utils
from scripts.predict import predict_match
from ipywidgets import widgets, VBox, HBox, Output
from IPython.display import display, HTML
import pandas as pd

print("Loading data...")
historical_data = data_loader.load_processed_data()
team_stats = utils.get_team_stats_table(historical_data)
team_to_league = utils.get_team_to_league_map(historical_data)
all_teams = sorted(utils.get_all_teams(historical_data))
t1_leagues = utils.get_available_leagues(tier=1)
t2_leagues = utils.get_available_leagues(tier=2)
print(f"Loaded {len(all_teams)} teams | {len(t1_leagues)} T1 leagues | {len(t2_leagues)} T2 leagues")

Loading data...
Loaded 695 teams | 47 T1 leagues | 47 T2 leagues


In [2]:
style = {'description_width': '100px'}

home_dropdown = widgets.Dropdown(
    options=all_teams, description='Home Team:',
    style=style, layout={'width': '400px'})
away_dropdown = widgets.Dropdown(
    options=all_teams, description='Away Team:',
    style=style, layout={'width': '400px'},
    value=all_teams[1] if len(all_teams) > 1 else None)

odds_h = widgets.FloatText(description='Odds H:', value=0, style=style, layout={'width': '200px'})
odds_d = widgets.FloatText(description='Odds D:', value=0, style=style, layout={'width': '200px'})
odds_a = widgets.FloatText(description='Odds A:', value=0, style=style, layout={'width': '200px'})

output = Output()

def _market_table(title, rows):
    h = f"<h4 style='margin:10px 0 5px;'>{title}</h4>"
    h += "<table style='font-size:15px;border-collapse:collapse;'>"
    for i, (label, prob) in enumerate(rows):
        bg = "background:#ecf0f1;" if i % 2 == 0 else ""
        o = f"{1/prob:.2f}" if prob > 0.001 else "-"
        h += (f"<tr style='{bg}'>"
              f"<td style='padding:8px 14px;border:1px solid #bdc3c7;'><b>{label}</b></td>"
              f"<td style='padding:8px 14px;border:1px solid #bdc3c7;'>{prob:.1%}</td>"
              f"<td style='padding:8px 14px;border:1px solid #bdc3c7;'>@ {o}</td></tr>")
    return h + "</table>"

def _tier_html(t, tier_label):
    h = f"<h3 style='color:#34495e;border-bottom:2px solid #ecf0f1;padding-bottom:4px;'>{tier_label}</h3>"
    if '1x2' in t:
        p = t['1x2']
        h += _market_table('1X2', [('Home', p['home']), ('Draw', p['draw']), ('Away', p['away'])])
    if 'ou25' in t:
        o = t['ou25']
        h += _market_table('Over/Under 2.5', [('Over 2.5', o['over']), ('Under 2.5', o['under'])])
    if 'btts' in t:
        b = t['btts']
        h += _market_table('Both Teams To Score', [('Yes', b['yes']), ('No', b['no'])])
    if 'xg' in t:
        h += f"<h4 style='margin:10px 0 5px;'>Expected Goals</h4>"
        h += f"<p style='font-size:20px;'><b>{t['xg']['home']:.2f}</b> - <b>{t['xg']['away']:.2f}</b></p>"
        if 'top_scores' in t:
            h += "<h4 style='margin:10px 0 5px;'>Most Likely Scores</h4><table style='font-size:14px;border-collapse:collapse;'>"
            for sc, pr in t['top_scores'][:6]:
                h += f"<tr><td style='padding:4px 12px;border:1px solid #bdc3c7;'>{sc}</td><td style='padding:4px 12px;border:1px solid #bdc3c7;'>{pr:.1%}</td></tr>"
            h += "</table>"
    return h

def make_prediction(change):
    output.clear_output()
    with output:
        home_team, away_team = home_dropdown.value, away_dropdown.value
        if home_team == away_team:
            print("Please select different teams"); return
        odds = None
        if odds_h.value > 1 and odds_d.value > 1 and odds_a.value > 1:
            odds = {'H': odds_h.value, 'D': odds_d.value, 'A': odds_a.value}
        try:
            pred = predict_match(home_team, away_team, team_stats,
                                 team_to_league, historical_data, odds=odds)
            html = (f"<h2 style='color:#2c3e50;'>{home_team} vs {away_team}</h2>"
                    f"<p><b>League:</b> {pred['league']}</p>")
            html += _tier_html(pred, 'Tier 1 (No Odds)')
            if 'tier2' in pred:
                html += "<hr style='margin:20px 0;'>"
                html += _tier_html(pred['tier2'], 'Tier 2 (With Odds)')
            display(HTML(html))
        except Exception as e:
            print(f"Error: {e}")

for w in [home_dropdown, away_dropdown, odds_h, odds_d, odds_a]:
    w.observe(make_prediction, names='value')

odds_label = widgets.HTML("<p style='margin:8px 0 2px;color:#7f8c8d;'><em>Enter decimal odds > 1 for Tier 2 predictions (leave 0 to skip):</em></p>")
ui = VBox([home_dropdown, away_dropdown, odds_label, HBox([odds_h, odds_d, odds_a]), output])
display(ui)
make_prediction(None)